In [1]:
import pennylane as qml
from pennylane import numpy as qnp
# from pennylane.optimize import AdamOptimizer
from pennylane.templates import AmplitudeEmbedding, BasicEntanglerLayers

import sys
import os
import subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    !pip install -U ipython
    !git clone https://github.com/AdrianPanasiewicz/QML_for_radar_classification.git

    repo_url = "https://github.com/AdrianPanasiewicz/QML_for_radar_classification.git"
    repo_path = "/content/QML_for_radar_classification"
    colab_run_dir = os.path.join(repo_path, 'colab_run')

    def run(cmd, cwd=None):
        return subprocess.check_output(cmd, cwd=cwd, text=True).strip()

    if not os.path.isdir(os.path.join(repo_path, ".git")):
        run(["git", "clone", repo_url, repo_path])
    else:
        run(["git", "fetch", "origin"], cwd=repo_path)
        local_head = run(["git", "rev-parse", "HEAD"], cwd=repo_path)
        remote_head = run(["git", "rev-parse", "origin/HEAD"], cwd=repo_path)
        if local_head != remote_head:
            run(["git", "reset", "--hard", "origin/HEAD"], cwd=repo_path)

    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)


    os.makedirs(colab_run_dir, exist_ok=True)
    os.chdir(colab_run_dir)

    !pip install -q pennylane
    !pip install "ray[tune]"

from pathlib import Path
from torch import nn
from torch.optim import Adam
import torch
from tqdm import tqdm
from Data.Primitives.environment_classes import Drone, Radar, Context
from Data.Primitives.noise_models import AdditiveWhiteGaussianNoise
from Data.Primitives.presets import *
from Data.Generators.synthetic_dataset_generator import DatasetMetadata, DataRequest

from MachineLearning.Processing.file_loader import SyntheticDataFileLoader
from MachineLearning.Processing.frequency_domain_parser import FrequencyDomainDataParser
from MachineLearning.Processing.time_domain_parser import TimeDomainDataParser
from MachineLearning.Torch_datasets.synthetic_time_dataset import SyntheticTimeDomainRadarDataset
from MachineLearning.Torch_datasets.synthetic_frequency_dataset import SyntheticFrequencyDomainRadarDataset
from MachineLearning.Models.experiment_pure.classical_neural_network import ClassicalNeuralNetwork
from MachineLearning.Models.experiment_pure.quantum_neural_network import QuantumNeuralNetwork
from MachineLearning.Processing.data_visualizer import DataVisualizer
from MachineLearning.Trainers.statistical_trainer import TrainerForModelStatistics
from MachineLearning.Trainers.hyperparameter_trainer import TrainerForHyperparameterSearch
from MachineLearning.Trainers.abstract_trainer import AbstractTrainer

from torch.utils.data import DataLoader

ModuleNotFoundError: No module named 'pennylane'

In [ ]:
class QuantumNeuralNetwork(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.encodings = {
            "angle" : self.angle_encoding,
            "amplitude": self.amplitude_embedding
        }
        self.ansatzes = {
            "basic" : self.basic_ansatz,
        }

        self.init_kwargs = {
            "n_qubits": config['n_qubits'],
            "layers": config['layers'],
            "encoding": config["encoding"],
            "ansatz": config["ansatz"],
            "simulator": config["simulator"],
        }
        self.model_name = self.__class__.__name__

        self.dev = qml.device(self.init_kwargs["simulator"], wires=config["n_qubits"])

        @qml.qnode(self.dev, interface="torch")
        def classifier(inputs, weights):
            self.encodings[self.init_kwargs['encoding']](inputs)
            self.ansatzes[self.init_kwargs['ansatz']](weights)
            return qml.expval(qml.PauliZ(0))

        weight_shapes = {
            "weights": (self.init_kwargs["layers"], self.init_kwargs["n_qubits"], 3)
        }

        self.qnode = qml.qnn.TorchLayer(classifier, weight_shapes)


    def angle_encoding(self, x):
        n_qubits = self.init_kwargs['n_qubits']
        qml.AngleEmbedding(features=x, wires=range(n_qubits), rotation='X')
        for i in range(n_qubits-1):
            qml.CNOT(wires=[i, i+1])

    def amplitude_embedding(self, x):
        n_qubits = self.init_kwargs['n_qubits']
        qml.AmplitudeEmbedding(features=x, wires=range(n_qubits), pad_with=0.0, normalize=True)

    def basic_ansatz(self, weights):
        # n_layers, n_qubits = params.shape[0], params.shape[1]
        # qml.BasicEntanglerLayers(params, wires =range(n_qubits))
        n_layers, n_qubits = weights.shape[0], weights.shape[1]
        for i in range(n_layers):
            for j in range(n_qubits):
                qml.Rot(weights[i,j,0], weights[i,j,1], weights[i,j,2], wires=j)
            for j in range(n_qubits-1):
                qml.CNOT(wires=[j,j+1])

    def forward(self, x):
        return self.qnode(x)

In [ ]:
MODEL_REGISTRY = {
    "QuantumNeuralNetwork": QuantumNeuralNetwork,
}


class QuantumTrainer(AbstractTrainer):
    def __init__(self, training_path, validating_path, testing_path, criterion):
        super().__init__(training_path, validating_path, testing_path, criterion)

    # def return_config_from_ray_results(self, best_result, map_location="cpu"):
    #     checkpoint = best_result.checkpoint
    #     with checkpoint.as_directory() as checkpoint_dir:
    #         checkpoint_path = Path(checkpoint_dir) / "checkpoint.pt"
    #         checkpoint_data = torch.load(checkpoint_path, map_location=map_location)
    #
    #     model_class_name = checkpoint_data["model_class_name"]
    #     model_init_kwargs = checkpoint_data["model_init_kwargs"]
    #     net_state_dict = checkpoint_data["net_state_dict"]
    #
    #     # model_class = MODEL_REGISTRY[model_class_name]
    #     # model = model_class(**model_init_kwargs)
    #     # model.load_state_dict(net_state_dict)
    #     # model.eval()
    #     return model_class_name, model_init_kwargs, net_state_dict


    def train_model(self, model_class, config):
        training_config = config["training_config"]
        model_config = config["model_config"]

        device = training_config["device"]
        net = model_class(model_config).to(device)

        optimizer = Adam(net.parameters(), lr=training_config["lr"])
        trainloader = DataLoader(self.trainset, batch_size=int(training_config["batch_size"]), shuffle=True)
        valloader = DataLoader(self.valset, batch_size=int(training_config["batch_size"]), shuffle=False)

        data_dict = {
            "accuracy": [],
            "validation_loss": []
        }

        for epoch in tqdm(range(training_config["epochs"]), desc='Epochs'):
            net.train()

            for inputs, labels in trainloader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = net(inputs).squeeze()
                probs = (outputs + 1.0) / 2.0
                loss = self.criterion(probs, labels.float())
                loss.backward()
                optimizer.step()

            net.eval()
            val_loss = 0.0
            val_steps = 0
            total = 0
            correct = 0

            with torch.no_grad():
                for inputs, labels in valloader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = net(inputs).squeeze()
                    probs = (outputs + 1.0) / 2.0
                    loss = self.criterion(probs, labels.float())
                    val_loss += loss.item()
                    val_steps += 1

                    predicted = (outputs >=0).long()
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

                    print("Outputs: ", outputs[:10])
                    print("Probs: ", probs[:10])
                    print("Pred.: ", predicted[:10])
                    print("Corr.: ", labels[:10])


            data_dict["validation_loss"].append(val_loss / val_steps)
            data_dict["accuracy"].append(100 * correct / total)


        return data_dict

    def test_model(self, model):
        pass

In [ ]:
model_config = {
    "n_qubits"  : 10,
    "layers"    : 2,
    "encoding"  : "angle",
    "ansatz"    : "basic",
    "simulator" : 'lightning.qubit',
}
training_config = {
    "epochs" : 10,
    "lr" : 3e-4,
    "device" : "cpu",
    "batch_size" : 8,
}

config = {
    "model_config" : model_config,
    "training_config" : training_config,
}

qnn = QuantumNeuralNetwork(model_config)

x = torch.randn(10)

y = qnn.qnode(x)
fig, ax = qml.draw_mpl(qnn.qnode)(x)
fig.show()

In [ ]:
PROJECT_ROOT = Path().cwd().parent.parent
training_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "training_dataset.pkl"
validating_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "validating_dataset.pkl"
testing_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "testing_dataset.pkl"

trainer = QuantumTrainer(training_path, validating_path, testing_path, criterion=nn.BCELoss())

model_config = {
    "n_qubits"  : 10,
    "layers"    : 4,
    "encoding"  : "angle",
    "ansatz"    : "basic",
    "simulator" : 'lightning.qubit',
}
training_config = {
    "epochs" : 10,
    "lr" : 3e-3,
    "device" : "cpu",
    "batch_size" : 4
}

config = {
    "model_config" : model_config,
    "training_config" : training_config,
}

results = trainer.train_model(QuantumNeuralNetwork, config)

In [ ]:
plotter = DataVisualizer()
plotter.plot_statistics(results["accuracy"], torch.zeros(len(results["accuracy"])))